# Inspect modeling-ready data

Loads `candidate_detail.parquet`, `features/fingerprints.parquet`, and `features/molformer_embeddings.parquet` from a pipeline + featurization run, and joins them on `candidate_id` for manual inspection.

Edit `OUT_DIR` if your `--output` lives somewhere other than `docs/`.

In [17]:
from pathlib import Path
import numpy as np
import pandas as pd

OUT_DIR = Path('outputs')
CAND_PATH  = OUT_DIR / 'candidate_detail.parquet'
FP_PATH    = OUT_DIR / 'features' / 'fingerprints.parquet'
EMB_PATH   = OUT_DIR / 'features' / 'molformer_embeddings.parquet'

for p in (CAND_PATH, FP_PATH, EMB_PATH):
    print(f'{p}: {"OK" if p.exists() else "MISSING"}')

outputs/candidate_detail.parquet: OK
outputs/features/fingerprints.parquet: OK
outputs/features/molformer_embeddings.parquet: OK


In [18]:
TRIAL_PATH = OUT_DIR / 'trial_detail.parquet'

In [19]:
trials = pd.read_parquet(TRIAL_PATH)

In [20]:
candidates = pd.read_parquet(CAND_PATH)
print('shape:', candidates.shape)
print('canonical SMILES coverage:', candidates['smiles_canonical'].notna().sum(), '/', len(candidates))
print('standardization status counts:')
print(candidates['smiles_standardization_status'].value_counts(dropna=False))
candidates[['candidate_id', 'drug_name', 'indication', 'highest_phase', 'smiles', 'smiles_canonical', 'smiles_standardization_status']].head()

shape: (55350, 159)
canonical SMILES coverage: 55344 / 55350
standardization status counts:
smiles_standardization_status
ok              55344
failed_parse        6
Name: count, dtype: int64


,candidate_id,drug_name,indication,highest_phase,smiles,smiles_canonical,smiles_standardization_status
0,db:DB00986__postoperative nausea and vomiting,glycopyrronium,postoperative nausea and vomiting,Phase 4,C[N+]1(C)CCC(C1)OC(=O)C(O)(C1CCCC1)C1=CC=CC=C1,C[N+]1(C)CCC(OC(=O)C(O)(c2ccccc2)C2CCCC2)C1,ok
1,db:DB11881__lymphoma,auy922,lymphoma,Phase 2,CCNC(=O)C1=NOC(=C1C1=CC=C(CN2CCOCC2)C=C1)C1=CC...,CCNC(=O)c1noc(-c2cc(C(C)(C)C)c(O)cc2O)c1-c1ccc...,ok
2,db:DB04918__cephalosporins,ceftobiprole,cephalosporins,Phase 1,[H][C@@]1(NC(=O)C(=N/O)\C2=NSC(N)=N2)C(=O)N2C(...,Nc1nc(/C(=N/O)C(=O)N[C@@H]2C(=O)N3C(C(=O)O)=C(...,ok
3,"db:DB08880__multiple sclerosis, chronic progre...",teriflunomide,relapsing multiple sclerosis,Phase 3,C\C(O)=C(/C#N)C(=O)NC1=CC=C(C=C1)C(F)(F)F,C/C(O)=C(\C#N)C(=O)Nc1ccc(C(F)(F)F)cc1,ok
4,"db:DB08865__lymphoma, follicular",crizotinib,c-met positive gastric cancer,Phase 2,C[C@@H](OC1=CC(=CN=C1N)C1=CN(N=C1)C1CCNCC1)C1=...,C[C@@H](Oc1cc(-c2cnn(C3CCNCC3)c2)cnc1N)c1c(Cl)...,ok


In [21]:
fingerprints = pd.read_parquet(FP_PATH) if FP_PATH.exists() else pd.DataFrame()
print('shape:', fingerprints.shape)
if len(fingerprints):
    sample = fingerprints.iloc[0]
    print('ecfp4 length:', len(sample['ecfp4']), '(expected 2048) — popcount:', int(np.array(sample['ecfp4']).sum()))
    print('maccs length:', len(sample['maccs']), '(expected 167)  — popcount:', int(np.array(sample['maccs']).sum()))
fingerprints.head()

shape: (55344, 3)
ecfp4 length: 2048 (expected 2048) — popcount: 40
maccs length: 167 (expected 167)  — popcount: 49


,candidate_id,ecfp4,maccs
0,db:DB00986__postoperative nausea and vomiting,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,db:DB11881__lymphoma,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,db:DB04918__cephalosporins,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, ..."
3,"db:DB08880__multiple sclerosis, chronic progre...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,"db:DB08865__lymphoma, follicular","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [22]:
embeddings = pd.read_parquet(EMB_PATH) if EMB_PATH.exists() else pd.DataFrame()
print('shape:', embeddings.shape)
if len(embeddings):
    sample = np.array(embeddings.iloc[0]['embedding'])
    print('embedding dim:', sample.shape[0], '— mean:', float(sample.mean()), 'std:', float(sample.std()))
    print('model:', embeddings['model'].iloc[0])
embeddings.head()

shape: (55344, 3)
embedding dim: 768 — mean: -0.0019777358035639736 std: 0.5875132142199198
model: ibm/MoLFormer-XL-both-10pct


,candidate_id,embedding,model
0,db:DB00986__postoperative nausea and vomiting,"[0.5490396022796631, -0.09050367027521133, 0.4...",ibm/MoLFormer-XL-both-10pct
1,db:DB11881__lymphoma,"[-0.09139877557754517, 0.5208075046539307, 0.5...",ibm/MoLFormer-XL-both-10pct
2,db:DB04918__cephalosporins,"[0.4773001968860626, 0.20929238200187683, 0.92...",ibm/MoLFormer-XL-both-10pct
3,"db:DB08880__multiple sclerosis, chronic progre...","[0.4328056871891022, 0.42998093366622925, 0.93...",ibm/MoLFormer-XL-both-10pct
4,"db:DB08865__lymphoma, follicular","[0.43463554978370667, 1.0674208402633667, 0.46...",ibm/MoLFormer-XL-both-10pct


In [23]:
joined = candidates.merge(fingerprints, on='candidate_id', how='left') \
                   .merge(embeddings,   on='candidate_id', how='left')
print('joined shape:', joined.shape)
print('rows with all three:', ((joined['smiles_canonical'].notna()) & (joined['ecfp4'].notna()) & (joined['embedding'].notna())).sum())
joined[['candidate_id', 'drug_name', 'indication', 'smiles_canonical', 'ecfp4', 'embedding']].head()

joined shape: (55350, 163)
rows with all three: 55344


,candidate_id,drug_name,indication,smiles_canonical,ecfp4,embedding
0,db:DB00986__postoperative nausea and vomiting,glycopyrronium,postoperative nausea and vomiting,C[N+]1(C)CCC(OC(=O)C(O)(c2ccccc2)C2CCCC2)C1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.5490396022796631, -0.09050367027521133, 0.4..."
1,db:DB11881__lymphoma,auy922,lymphoma,CCNC(=O)c1noc(-c2cc(C(C)(C)C)c(O)cc2O)c1-c1ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.09139877557754517, 0.5208075046539307, 0.5..."
2,db:DB04918__cephalosporins,ceftobiprole,cephalosporins,Nc1nc(/C(=N/O)C(=O)N[C@@H]2C(=O)N3C(C(=O)O)=C(...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.4773001968860626, 0.20929238200187683, 0.92..."
3,"db:DB08880__multiple sclerosis, chronic progre...",teriflunomide,relapsing multiple sclerosis,C/C(O)=C(\C#N)C(=O)Nc1ccc(C(F)(F)F)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.4328056871891022, 0.42998093366622925, 0.93..."
4,"db:DB08865__lymphoma, follicular",crizotinib,c-met positive gastric cancer,C[C@@H](Oc1cc(-c2cnn(C3CCNCC3)c2)cnc1N)c1c(Cl)...,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.43463554978370667, 1.0674208402633667, 0.46..."


In [24]:
joined.iloc[0]

candidate_id          db:DB00986__postoperative nausea and vomiting
drug_name                                            glycopyrronium
drug_name_raw                                        Glycopyrronium
indication                        postoperative nausea and vomiting
highest_phase                                               Phase 4
                                        ...                        
outcomes_agree                                                 None
ecfp4             [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
maccs             [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
embedding         [0.5490396022796631, -0.09050367027521133, 0.4...
model                                   ibm/MoLFormer-XL-both-10pct
Name: 0, Length: 163, dtype: object

In [25]:
trials

,nct_id,candidate_id,also_in_candidate_ids,candidate_drug,candidate_drug_raw,candidate_indication,candidate_modality,candidate_disease_area,candidate_outcome,candidate_highest_phase,...,trial_inferred_label,trial_mesh_intervention_terms,trial_mesh_condition_terms,trial_mesh_condition_tree_numbers,trial_start_date,trial_completion_date,trial_last_update_submitted_date,trial_is_single_arm,trial_sponsor,trial_title
0,NCT05265507,db:DB00986__postoperative nausea and vomiting,[],glycopyrronium,Glycopyrronium,postoperative nausea and vomiting,unknown,other,Failed Phase 3,Phase 4,...,NaN,"[Glycopyrrolate, Ondansetron]",[Postoperative Nausea and Vomiting],"[C23.550.767.859, C23.888.821.712.700, C23.888...",2022-03-21,2023-05-30,2023-02-22,False,The Second Affiliated Hospital of Chongqing Me...,Comparison of Postoperative Anti-nausea and Vo...
1,NCT01485536,db:DB11881__lymphoma,[],auy922,AUY922,lymphoma,unknown,unknown,Failed Phase 2,Phase 2,...,0.0,"[5-(2,4-dihydroxy-5-isopropylphenyl)-4-(4-morp...",[Lymphoma],"[C04.557.386, C15.604.515.569, C20.683.515.761]",2012-08-31,2015-11-30,2016-11-28,True,M.D. Anderson Cancer Center,A Study of the HSP90 Inhibitor AUY922
2,NCT00965042,db:DB04918__cephalosporins,"[db:DB04918__drug resistance, db:DB04918__anti...",ceftobiprole,Ceftobiprole,cephalosporins,unknown,unknown,Failed Phase 1,Phase 1,...,0.0,[ceftobiprole],[],[],2009-04-30,2009-06-30,2012-07-27,False,Basilea Pharmaceutica,Effect of Ceftobiprole on Human Intestinal Mic...
3,NCT06372145,"db:DB08880__multiple sclerosis, chronic progre...","[db:DB08880__multiple sclerosis, chronic progr...",teriflunomide,Teriflunomide,relapsing multiple sclerosis,unknown,neurology,Approved,Phase 3,...,1.0,[teriflunomide],"[Multiple Sclerosis, Chronic Progressive]","[C10.114.375.500.200, C10.314.350.500.200, C20...",2024-04-16,2029-04-30,2026-02-12,False,Sanofi,A Study to Investigate Long-term Safety and To...
4,NCT02435108,"db:DB08865__lymphoma, follicular",[],crizotinib,crizotinib,c-met positive gastric cancer,unknown,unknown,Failed Phase 2,Phase 2,...,0.0,[Crizotinib],"[Lymphoma, Follicular]","[C04.557.386.480.350, C15.604.515.569.480.350,...",2014-05-15,2016-07-15,2017-02-16,True,Samsung Medical Center,A Pilot Study of Crizotinib in Patients With c...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58132,NCT04257929,db:DB11642__prader-willi syndrome,[],pitolisant oral tablets,Pitolisant oral tablets,prader-willi syndrome,unknown,unknown,Ongoing,Phase 3,...,NaN,[pitolisant],[Prader-Willi Syndrome],"[C10.597.606.360.690, C16.131.077.730, C16.131...",2020-12-09,2028-09-07,2026-02-05,False,"Harmony Biosciences Management, Inc.",A Phase 2 Study to Evaluate the Safety and Eff...
58133,NCT07219485,db:DB11642__prader-willi syndrome,[],pitolisant oral tablets,Pitolisant oral tablets,prader-willi syndrome,unknown,unknown,Ongoing,Phase 3,...,NaN,[pitolisant],[Prader-Willi Syndrome],"[C10.597.606.360.690, C16.131.077.730, C16.131...",2025-08-20,2030-08-31,2026-03-13,True,"Harmony Biosciences Management, Inc.",A Study of Pitolisant in Participants With Pra...
58134,NCT03115177,db:DB01327__osteoarthritis,[],cefazolin,Cefazolin,osteoarthritis,unknown,other,Failed Phase 3,Phase 4,...,NaN,"[Cefazolin, Doxycycline]",[Osteoarthritis],"[C05.550.114.606, C05.799.613]",2015-11-30,2017-02-28,2018-05-16,False,Rush University Medical Center,Can Addition of Doxycycline Perioperatively Re...
58135,NCT02768337,db:DB08916__brain neoplasms,"[db:DB08916__brain neoplasms, db:DB08916__brai...",afatinib,Afatinib,lung cancer,unknown,oncology,Approved,Phase 2,...,0.0,[Afatinib],"[Brain Neoplasms, Breast Neoplasms, Lung Neopl...","[C04.588.180, C04.588.614.250.195, C04.588.894...",2015-03-10,2021-08-12,2024-01-30,False,Cambridge University Hospitals NHS Foundation ...,Cambridge Brain Mets Trial 1


In [27]:
len(trials.candidate_id.unique())

37997

In [28]:
len(candidates.candidate_id.unique())

55350

In [26]:
candidates

,candidate_id,drug_name,drug_name_raw,indication,highest_phase,trial_count,trial_ids,sponsors,earliest_start_date,latest_completion_date,...,llm_direct_evidence_sources,llm_direct_approval_date,llm_direct_commercialization_date,fda_timeline_outcome,fda_timeline_confidence,fda_timeline_reasoning,fda_timeline_evidence_sources,fda_timeline_approval_date,fda_timeline_commercialization_date,outcomes_agree
0,db:DB00986__postoperative nausea and vomiting,glycopyrronium,Glycopyrronium,postoperative nausea and vomiting,Phase 4,1,[NCT05265507],[The Second Affiliated Hospital of Chongqing M...,2022-03-21,2023-05-30,...,[],None,None,NaN,NaN,NaN,[],None,None,None
1,db:DB11881__lymphoma,auy922,AUY922,lymphoma,Phase 2,1,[NCT01485536],[M.D. Anderson Cancer Center],2012-08-31,2015-11-30,...,[],None,None,NaN,NaN,NaN,[],None,None,None
2,db:DB04918__cephalosporins,ceftobiprole,Ceftobiprole,cephalosporins,Phase 1,1,[NCT00965042],[Basilea Pharmaceutica],2009-04-30,2009-06-30,...,[],None,None,NaN,NaN,NaN,[],None,None,None
3,"db:DB08880__multiple sclerosis, chronic progre...",teriflunomide,Teriflunomide,relapsing multiple sclerosis,Phase 3,3,"[NCT06372145, NCT06372145, NCT06372145]",[Sanofi],2024-04-16,2029-04-30,...,[],None,None,NaN,NaN,NaN,[],None,None,None
4,"db:DB08865__lymphoma, follicular",crizotinib,crizotinib,c-met positive gastric cancer,Phase 2,1,[NCT02435108],[Samsung Medical Center],2014-05-15,2016-07-15,...,[],None,None,NaN,NaN,NaN,[],None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55345,db:DB00958__stage ic uterine sarcoma,carboplatin,carboplatin,stage ic uterine sarcoma,N/A,1,[NCT01367301],[Albert Einstein College of Medicine],2011-07-08,2018-08-15,...,[],None,None,NaN,NaN,NaN,[],None,None,None
55346,db:DB01233__enteral feeding intolerance (efi),metoclopramide,Metoclopramide,enteral feeding intolerance (efi),Phase 2,1,[NCT02784392],[Lyric Pharmaceuticals],2016-10-31,2018-03-31,...,[],None,None,NaN,NaN,NaN,[],None,None,None
55347,db:DB08916__brain neoplasms,afatinib,Afatinib,lung cancer,Phase 2,7,"[NCT02768337, NCT02768337, NCT02768337, NCT027...",[Cambridge University Hospitals NHS Foundation...,2015-03-10,2024-03-31,...,[],None,None,NaN,NaN,NaN,[],None,None,None
55348,db:DB14533__helicobacter pylori infections,zinc,zinc sulfate,helicobacter pylori infections,Phase 3,1,[NCT07275827],[Tanta University],2026-02-01,2026-10-31,...,[],None,None,NaN,NaN,NaN,[],None,None,None
